# Module 6 — Machine Learning Classifier
## Predicting Drug-likeness from Molecular Descriptors

This notebook uses **scikit-learn** to train a machine learning model that predicts whether an endophyte-derived compound is drug-like based on its molecular descriptors.

**What we do:**
- Use Lipinski descriptors (MW, LogP, HBD, HBA) as **features**
- Use Lipinski pass/fail as the **target label**
- Train a **Random Forest classifier**
- Evaluate using confusion matrix, accuracy, and feature importance
- Visualise results

**This notebook is independent — no other notebook needs to run first.**

In [ ]:
# ── CELL 1: Install libraries ──
!pip install rdkit scikit-learn matplotlib seaborn --quiet
print("✓ Libraries installed")

In [ ]:
# ── CELL 2: Import libraries ──
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import os

from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, ConfusionMatrixDisplay
)
from sklearn.preprocessing import LabelEncoder

os.makedirs("figures", exist_ok=True)
print("✓ Libraries imported")

In [ ]:
# ── CELL 3: Define compound dataset with SMILES ──
# Extended dataset with known drug-like and non-drug-like compounds
# to give the ML model enough data to learn from

compounds = [
    # Endophyte-derived compounds (from NTCC review)
    {"name": "Taxol",           "smiles": "CC1=C2C(C(=O)C3(C(CC4C(C3C(C(=C2OC1=O)C)(C)C)OC(=O)C5=CC=CC=C5)OC(=O)C)O)OC(=O)C6=CC=CC=C6", "source": "Endophyte"},
    {"name": "Camptothecin",    "smiles": "CCC1(O)C(=O)OCC2=C1C=C1N=C3C=CC=CC3=CC1=C2", "source": "Endophyte"},
    {"name": "Podophyllotoxin", "smiles": "COC1=C(OC)C(=CC2C3CC(=O)C4=CC(OC)=C(OC)C(OC)=C4C3OC2=O)C=C1", "source": "Endophyte"},
    {"name": "Griseofulvin",    "smiles": "COC1=CC(=O)CC(C)C12OC1=C(Cl)C(=O)C(OC)=CC1=C2", "source": "Endophyte"},
    {"name": "Brefeldin A",     "smiles": "CCCCCCC1CC(O)CC(=O)OCCCCC=CC1O", "source": "Endophyte"},
    {"name": "Huperzine A",     "smiles": "CC1=CC2CC(=C)C(N)=CC2=CC1=O", "source": "Endophyte"},
    {"name": "Emodin",          "smiles": "CC1=CC2=CC(=CC(=O)C2=C(O)C1=O)O", "source": "Endophyte"},
    {"name": "Pestacin",        "smiles": "OC1OC2=CC=CC=C2CC1O", "source": "Endophyte"},
    {"name": "Subglutinol A",   "smiles": "CC1(C)CCC(O)C2(C)CCC(=O)C=C12", "source": "Endophyte"},
    {"name": "Helvolic acid",   "smiles": "CC(C)(C)C1CCC2(C(=O)O)CCC3C2C1CCC3OC(C)=O", "source": "Endophyte"},
    {"name": "Hypericin",       "smiles": "CC1=CC2=C(C(O)=C1)C1=C(C=C3C(O)=CC(C)=CC3=C1O)C2=O", "source": "Endophyte"},
    {"name": "Ergoflavin",      "smiles": "COC1=C2C(=O)C3=C(OC)C(OC)=C(OC)C=C3C(=O)C2=C(OC)C(OC)=C1OC", "source": "Endophyte"},
    # Well-known approved oral drugs (drug-like)
    {"name": "Aspirin",         "smiles": "CC(=O)Oc1ccccc1C(=O)O", "source": "Reference"},
    {"name": "Ibuprofen",       "smiles": "CC(C)Cc1ccc(cc1)C(C)C(=O)O", "source": "Reference"},
    {"name": "Paracetamol",     "smiles": "CC(=O)Nc1ccc(O)cc1", "source": "Reference"},
    {"name": "Caffeine",        "smiles": "Cn1c(=O)c2c(ncn2C)n(c1=O)C", "source": "Reference"},
    {"name": "Metformin",       "smiles": "CN(C)C(=N)NC(=N)N", "source": "Reference"},
    {"name": "Ciprofloxacin",   "smiles": "O=C(O)c1cn(C2CC2)c2cc(N3CCNCC3)c(F)cc2c1=O", "source": "Reference"},
    {"name": "Amoxicillin",     "smiles": "CC1(C)SC2C(NC(=O)C(N)c3ccc(O)cc3)C(=O)N2C1C(=O)O", "source": "Reference"},
    {"name": "Atorvastatin",    "smiles": "CC(C)c1n(CC(O)CC(O)CC(=O)O)c(C(=O)Nc2ccccc2F)c(-c2ccccc2)c1-c1ccc(F)cc1", "source": "Reference"},
    {"name": "Omeprazole",      "smiles": "COc1ccc2nc(S(=O)Cc3ncc(C)c(OC)c3C)[nH]c2c1", "source": "Reference"},
    {"name": "Warfarin",        "smiles": "OC(CC(=O)c1ccccc1)c1c(=O)oc2ccccc12", "source": "Reference"},
    # Known non-drug-like large molecules
    {"name": "Rapamycin",       "smiles": "CCC(=O)O[C@@H]1C[C@H](CC[C@@H]([C@@H]([C@H](C(=O)[C@@H](CC(=O)[C@@H]([C@@H]([C@H]([C@@H](CC1=O)C)O)OC)C)C)C)O)OC)OC", "source": "Reference"},
    {"name": "Vancomycin",      "smiles": "CNC(CC1=CC(Cl)=C(OC2=C(OC3=CC=C(C[C@@H](NC(=O)[C@@H]4CC(=O)N[C@@H](C(=O)N[C@H](C(=O)O)C5=CC=C(O)C(Cl)=C5)C6=C(O)C=CC(=C6)[C@H](NC4=O)O)C3=O)C=C2Cl)C=C1)C(=O)O", "source": "Reference"},
    {"name": "Cyclosporine",    "smiles": "CCC1C(=O)N(CC(=O)N(C(C(=O)NC(C(=O)N(C(C(=O)NC(C(=O)NC(C(=O)N(C(C(=O)N(C(C(=O)N(C(C(=O)N1C)CC(C)C)C)CC(C)C)C)C)CC(C)C)C)C)C)CC(C)C)C)C)C", "source": "Reference"},
]

df = pd.DataFrame(compounds)
print(f"✓ Dataset: {len(df)} compounds")
df.head()

In [ ]:
# ── CELL 4: Compute Lipinski descriptors ──
def compute_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return {"MW": None, "LogP": None, "HBD": None, "HBA": None}
    return {
        "MW":   round(Descriptors.ExactMolWt(mol), 2),
        "LogP": round(Descriptors.MolLogP(mol), 2),
        "HBD":  rdMolDescriptors.CalcNumHBD(mol),
        "HBA":  rdMolDescriptors.CalcNumHBA(mol)
    }

desc = df["smiles"].apply(compute_descriptors)
df_desc = pd.DataFrame(desc.tolist())
df = pd.concat([df, df_desc], axis=1).dropna()

# Apply Lipinski rule to create target label
def lipinski_label(row):
    violations = sum([
        row["MW"]   > 500,
        row["LogP"] > 5,
        row["HBD"]  > 5,
        row["HBA"]  > 10
    ])
    return "Drug-like" if violations <= 1 else "Non-drug-like"

df["label"] = df.apply(lipinski_label, axis=1)

print("✓ Descriptors computed")
print(f"\nLabel distribution:")
print(df["label"].value_counts())
df[["name", "MW", "LogP", "HBD", "HBA", "label"]]

In [ ]:
# ── CELL 5: Train Random Forest classifier ──
features = ["MW", "LogP", "HBD", "HBA"]
X = df[features]
y = df["label"]

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# Train Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

# Predictions
y_pred = rf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"✓ Model trained!")
print(f"  Training set size: {len(X_train)}")
print(f"  Test set size:     {len(X_test)}")
print(f"  Accuracy:          {accuracy:.0%}")
print()
print("Classification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
# ── CELL 6: Cross-validation (more robust evaluation) ──
cv_scores = cross_val_score(rf, X, y, cv=5, scoring="accuracy")
print(f"5-Fold Cross-Validation Accuracy:")
print(f"  Scores: {[f'{s:.0%}' for s in cv_scores]}")
print(f"  Mean:   {cv_scores.mean():.0%}")
print(f"  Std:    ±{cv_scores.std():.0%}")

In [ ]:
# ── CELL 7: Plot confusion matrix ──
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred, labels=rf.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=rf.classes_)
disp.plot(ax=axes[0], colorbar=False, cmap="Blues")
axes[0].set_title("Confusion Matrix", fontweight="bold", fontsize=12)

# Feature importance
importances = rf.feature_importances_
feat_df = pd.DataFrame({"Feature": features, "Importance": importances})
feat_df = feat_df.sort_values("Importance", ascending=True)

colors = ["#1565C0", "#1976D2", "#42A5F5", "#90CAF9"]
axes[1].barh(feat_df["Feature"], feat_df["Importance"],
             color=colors, edgecolor="white")
axes[1].set_title("Feature Importance\n(Which descriptor matters most?)",
                  fontweight="bold", fontsize=12)
axes[1].set_xlabel("Importance Score")
axes[1].spines[["top", "right"]].set_visible(False)

for i, (val, name) in enumerate(zip(feat_df["Importance"], feat_df["Feature"])):
    axes[1].text(val + 0.005, i, f"{val:.3f}", va="center", fontsize=10)

plt.suptitle("Random Forest — Drug-likeness Classifier",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("figures/ml_classifier.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Figure saved to figures/ml_classifier.png")

In [ ]:
# ── CELL 8: Predict on all endophyte compounds ──
# Now use the trained model to predict drug-likeness
# for ALL endophyte compounds in our dataset

df_endophyte = df[df["source"] == "Endophyte"].copy()
df_endophyte["predicted"] = rf.predict(df_endophyte[features])
df_endophyte["confidence"] = rf.predict_proba(df_endophyte[features]).max(axis=1)
df_endophyte["confidence"] = df_endophyte["confidence"].apply(lambda x: f"{x:.0%}")

print("=== ML Predictions for Endophyte Compounds ===")
print()
for _, row in df_endophyte.iterrows():
    icon = "✅" if row["predicted"] == "Drug-like" else "❌"
    print(f"{icon} {row['name']:<20} → {row['predicted']:<15} (confidence: {row['confidence']})")

In [ ]:
# ── CELL 9: Download figure ──
from google.colab import files
files.download("figures/ml_classifier.png")
print("✓ Downloaded! Upload to your GitHub repo under figures/")

## Summary

This module demonstrates a complete machine learning workflow:
1. **Feature engineering** — computed molecular descriptors from SMILES using RDKit
2. **Model training** — Random Forest classifier on labelled compound data
3. **Evaluation** — confusion matrix, classification report, cross-validation
4. **Prediction** — applied the trained model to endophyte-specific compounds
5. **Interpretation** — feature importance plot shows which molecular property matters most for drug-likeness

**Key insight:** The most important feature for predicting drug-likeness is typically **Molecular Weight** — compounds that are too large struggle to be absorbed orally, which is exactly what Lipinski's rules predict.